In [1]:
# 1. Install required packages (run once in your environment / notebook)
#!pip install gensim bertopic sentence_transformers umap-learn hdbscan spacy wordcloud

from bertopic import BERTopic

from sentence_transformers import SentenceTransformer

from sklearn.feature_extraction.text import CountVectorizer

from gensim.models.coherencemodel import CoherenceModel

import gensim

# spaCy for better lemmatization
import spacy

import gensim.corpora as corpora


# Optional: suppress warnings for cleaner output
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [2]:
# ============================================================
# REPRODUCIBILITY SETTINGS
# ============================================================

import random
import numpy as np

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print(f"Random seed set to: {SEED}")

Random seed set to: 42


In [3]:
# 2. Load example data 
import pandas as pd

# Load the Amazon Electronics Reviews dataset
df = pd.read_csv("../data/processed/cleaned_reviews_latest.csv")

# Remove rows with missing review text
df = df.dropna(subset=["text"])

# Combine review title and review text (recommended)
df["review"] = df["title"].fillna("") + " " + df["text"].fillna("")

# Convert reviews to a list (same format as the demo)
data = df["review"].tolist()

print(f"Number of documents: {len(data)}")
print("Example document snippet:\n", data[0][:300], "...\n")

Number of documents: 10518
Example document snippet:
 Smells like gasoline! Going back! First & most offensive: they reek of gasoline so if you are sensitive/allergic to petroleum products like I am you will want to pass on these.  Second: the phone adapter is useless as-is. Mine was not drilled far enough to be able to tighten it into place for my iPh ...



In [4]:
# 3. Preprocessing functions
# load spacy's small lanuage model that includes Tokenizer; POS tagger (part-of-speech);
# Lemmatizer; Dependency parser; Named Entity Recognizer (NER) without building dependency tree and
# detecting and labelling named entities (performance)

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def sent_to_words(sentences):
    """Tokenize and clean basic text"""
    for sentence in sentences:
        yield gensim.utils.simple_preprocess(str(sentence), deacc=True)  # lowercase, remove punct

def remove_stopwords(texts):
    return [[word for word in gensim.utils.simple_preprocess(" ".join(doc))
             if word not in gensim.parsing.preprocessing.STOPWORDS]
            for doc in texts]

# specifying the words by their part-of-speech / grammatical category) that should be kept
def lemmatization(texts, allowed_postags=["NOUN", "ADJ", "VERB", "ADV"]):
    """Lemmatize with spaCy"""
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent))
        texts_out.append([token.lemma_ for token in doc
                          if token.pos_ in allowed_postags])
    return texts_out

# Apply preprocessing pipeline
data_words = list(sent_to_words(data))
data_words = remove_stopwords(data_words)
data_lemmatized = lemmatization(data_words)

print("\nExample after lemmatization:", data_lemmatized[0][:15])

documents = [
    " ".join(doc)
    for doc in data_lemmatized
]


Example after lemmatization: ['smell', 'gasoline', 'go', 'offensive', 'reek', 'gasoline', 'sensitive', 'allergic', 'petroleum', 'product', 'want', 'pass', 'second', 'phone', 'adapter']


In [5]:
# 4. Create Dictionary & Documents

# Create a Gensim dictionary.
# BERTopic itself does not use this dictionary during training,
# but it is required later for computing the topic coherence score.

id2word = corpora.Dictionary(data_lemmatized)

id2word.filter_extremes(
    no_below=5,
    no_above=0.40
)

# Convert tokenized reviews back into text strings.
# BERTopic expects a list of complete documents rather than token lists.

documents = [
    " ".join(doc)
    for doc in data_lemmatized
]

In [6]:
# ============================================================
# 5. Train DEFAULT BERTopic Model
# ============================================================

import time
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel


# ============================================================
# 5.1 Sentence-BERT Embedding Model
# ============================================================

# SentenceTransformer converts each review into a dense
# semantic vector that captures contextual meaning.

default_embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


# ============================================================
# 5.2 Vectorizer Model
# ============================================================

# Used to generate the c-TF-IDF representation for extracting
# representative topic words.
#
# Both unigrams and bigrams are included.

default_vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)


# ============================================================
# 5.3 Default BERTopic Model
# ============================================================

# The default model uses BERTopic's default UMAP and HDBSCAN
# configurations.
#
# Pipeline:
#
# Sentence-BERT
#      ↓
# Default UMAP
#      ↓
# Default HDBSCAN
#      ↓
# c-TF-IDF
#      ↓
# Topics

default_topic_model = BERTopic(

    embedding_model=default_embedding_model,

    vectorizer_model=default_vectorizer_model,

    language="english",

    calculate_probabilities=True,

    verbose=True
)


# ============================================================
# 5.4 Train Default BERTopic Model
# ============================================================

start_time = time.time()

default_topics, default_probabilities = (
    default_topic_model.fit_transform(documents)
)

default_training_time = time.time() - start_time


# ============================================================
# 5.5 Topic Information
# ============================================================

default_topic_info = (
    default_topic_model.get_topic_info()
)


# Exclude topic -1 because this represents outlier documents

default_valid_topics = default_topic_info[
    default_topic_info["Topic"] != -1
]

default_number_of_topics = len(
    default_valid_topics
)


# ============================================================
# 5.6 Outlier Calculation
# ============================================================

default_outlier_count = sum(
    topic == -1
    for topic in default_topics
)

default_outlier_percentage = (
    default_outlier_count / len(documents)
) * 100


# ============================================================
# 5.7 Vocabulary Size
# ============================================================

default_vocabulary_size = len(
    default_topic_model.vectorizer_model.vocabulary_
)


# ============================================================
# 5.8 Calculate c_v Coherence
# ============================================================

# Tokenise the original documents for coherence calculation

default_tokenized_documents = [
    document.lower().split()
    for document in documents
]


# Create Gensim dictionary

default_dictionary = Dictionary(
    default_tokenized_documents
)


# Extract the top words for each topic

default_topic_words = []

for topic_id in default_valid_topics["Topic"]:

    words = default_topic_model.get_topic(topic_id)

    if words is not None:

        # Select the top 10 representative words
        top_words = [
            word
            for word, score in words[:10]
        ]

        default_topic_words.append(top_words)


# Calculate c_v coherence

default_coherence_model = CoherenceModel(
    topics=default_topic_words,
    texts=default_tokenized_documents,
    dictionary=default_dictionary,
    coherence="c_v"
)

default_coherence_score = (
    default_coherence_model.get_coherence()
)


# ============================================================
# 5.9 Display Model Performance
# ============================================================

print("\n" + "=" * 65)
print("DEFAULT BERTopic MODEL RESULTS")
print("=" * 65)

print(
    f"Number of Topics:       "
    f"{default_number_of_topics}"
)

print(
    f"Coherence Score (c_v):  "
    f"{default_coherence_score:.4f}"
)

print(
    f"Outlier Documents:      "
    f"{default_outlier_count}"
)

print(
    f"Outlier Percentage:     "
    f"{default_outlier_percentage:.2f}%"
)

print(
    f"Vocabulary Size:        "
    f"{default_vocabulary_size}"
)

print(
    f"Training Time:           "
    f"{default_training_time:.2f} seconds"
)

print(
    f"Training Time:           "
    f"{default_training_time / 60:.2f} minutes"
)


# ============================================================
# 5.10 Display Topic Information
# ============================================================

print("\n" + "=" * 65)
print("DISCOVERED TOPICS")
print("=" * 65)

print(
    default_topic_info.to_string(index=False)
)


# ============================================================
# 5.11 Topic Interpretation
# ============================================================

print("\n" + "=" * 65)
print("TOPIC INTERPRETATION")
print("=" * 65)


# Create a table containing topic number,
# document count and representative words.

topic_interpretation = []

for topic_id in default_valid_topics["Topic"]:

    # Get topic words
    words = default_topic_model.get_topic(topic_id)

    # Get number of documents in topic
    topic_size = int(
        default_valid_topics.loc[
            default_valid_topics["Topic"] == topic_id,
            "Count"
        ].iloc[0]
    )

    # Extract top 10 words
    top_words = [
        word
        for word, score in words[:10]
    ]

    topic_interpretation.append({

        "Topic": topic_id,

        "Document Count": topic_size,

        "Top Words": ", ".join(top_words),

        "Interpretation": "Review top words and assign semantic label"

    })


# Convert to DataFrame

topic_interpretation_df = pd.DataFrame(
    topic_interpretation
)


print(
    topic_interpretation_df.to_string(
        index=False
    )
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-08-18 10:09:21,068 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/329 [00:00<?, ?it/s]

2026-08-18 10:14:26,048 - BERTopic - Embedding - Completed ✓
2026-08-18 10:14:26,050 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-18 10:16:09,076 - BERTopic - Dimensionality - Completed ✓
2026-08-18 10:16:09,078 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-18 10:16:29,864 - BERTopic - Cluster - Completed ✓
2026-08-18 10:16:31,432 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-18 10:16:36,200 - BERTopic - Representation - Completed ✓



DEFAULT BERTopic MODEL RESULTS
Number of Topics:       138
Coherence Score (c_v):  0.6150
Outlier Documents:      3510
Outlier Percentage:     33.37%
Vocabulary Size:        203593
Training Time:           438.23 seconds
Training Time:           7.30 minutes

DISCOVERED TOPICS
 Topic  Count                                                         Name                                                                                                                                            Representation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [8]:
# ============================================================
# 5. Train Improved BERTopic Model
# ============================================================

import time

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

from umap import UMAP
from hdbscan import HDBSCAN

from bertopic import BERTopic


# ============================================================
# 5.1 Sentence-BERT Embedding Model
# ============================================================

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


# ============================================================
# 5.2 UMAP Dimensionality Reduction
# ============================================================

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)


# ============================================================
# 5.3 HDBSCAN Clustering
# ============================================================

hdbscan_model = HDBSCAN(
    min_cluster_size=100,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)


# ============================================================
# 5.4 CountVectorizer
# ============================================================

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5
)


# ============================================================
# 5.5 BERTopic Model
# ============================================================

improved_topic_model = BERTopic(

    # Sentence-BERT embeddings
    embedding_model=embedding_model,

    # UMAP
    umap_model=umap_model,

    # HDBSCAN
    hdbscan_model=hdbscan_model,

    # c-TF-IDF representation
    vectorizer_model=vectorizer_model,

    language="english",

    # Reduce the final number of topics
    nr_topics=30,

    calculate_probabilities=False,

    verbose=True
)


# ============================================================
# 5.6 Train BERTopic
# ============================================================

start_time = time.time()

topics, probabilities = improved_topic_model.fit_transform(documents)

training_time = time.time() - start_time


# ============================================================
# 5.7 Model Performance
# ============================================================

topic_info = improved_topic_model.get_topic_info()

# Remove outlier topic (-1)
valid_topics = topic_info[
    topic_info["Topic"] != -1
]

number_of_topics = len(valid_topics)

# Count outliers
outlier_count = sum(
    topic == -1 for topic in topics
)

outlier_percentage = (
    outlier_count / len(documents)
) * 100


# ============================================================
# 5.8 Vocabulary Size
# ============================================================

vocabulary_size = len(
    improved_topic_model.vectorizer_model.vocabulary_
)


# ============================================================
# 5.9 Print Results
# ============================================================

print("\n" + "=" * 60)
print("BERTopic MODEL RESULTS")
print("=" * 60)

print(f"Number of topics: {number_of_topics}")
print(f"Outliers: {outlier_count}")
print(f"Outlier percentage: {outlier_percentage:.2f}%")
print(f"Vocabulary size: {vocabulary_size}")
print(f"Training time: {training_time:.2f} seconds")
print(f"Training time: {training_time / 60:.2f} minutes")

print("\nTopic Information:")
print(topic_info)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-08-18 10:24:07,055 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/329 [00:00<?, ?it/s]

2026-08-18 10:28:49,364 - BERTopic - Embedding - Completed ✓
2026-08-18 10:28:49,366 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-18 10:29:11,124 - BERTopic - Dimensionality - Completed ✓
2026-08-18 10:29:11,127 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-18 10:29:12,583 - BERTopic - Cluster - Completed ✓
2026-08-18 10:29:12,590 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-08-18 10:29:14,459 - BERTopic - Representation - Completed ✓
2026-08-18 10:29:14,463 - BERTopic - Topic reduction - Reducing number of topics
2026-08-18 10:29:14,465 - BERTopic - Topic reduction - Number of topics (30) is equal or higher than the clustered topics(27).
2026-08-18 10:29:14,468 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-18 10:29:17,968 - BERTopic - Representation - Completed ✓



BERTopic MODEL RESULTS
Number of topics: 26
Outliers: 2167
Outlier percentage: 20.60%
Vocabulary size: 6150
Training time: 312.18 seconds
Training time: 5.20 minutes

Topic Information:
    Topic  Count                                        Name  \
0      -1   2167                      -1_work_great_use_case   
1       0    172                     0_watch_band_wrist_wear   
2       1    115                         1_alexa_echo_dot_ad   
3       2   1577               2_sound_headphone_speaker_ear   
4       3    106                          3_fan_cool_air_pad   
5       4    641                 4_tv_remote_antenna_channel   
6       5    233                5_router_network_wifi_device   
7       6    504       6_star_star great_star work_star good   
8       7   1225                 7_cable_charge_cord_charger   
9       8    209             8_laptop_monitor_screen_graphic   
10      9    124                      9_port_hub_adapter_usb   
11     10    315                  10_drive_ca

In [9]:
#CALCULATE ACTUAL NUMBER OF TOPICS
num_topics = len(
    topic_info[topic_info["Topic"] != -1]
)

print("Number of discovered topics:", num_topics)

Number of discovered topics: 26


In [10]:
#CALCULATE OUTLIERS
outlier_count = sum(
    topic == -1 for topic in topics
)

total_documents = len(topics)

outlier_percentage = (
    outlier_count / total_documents
) * 100

print("Total documents:", total_documents)
print("Outlier documents:", outlier_count)
print("Outlier percentage:", round(outlier_percentage, 2), "%")

Total documents: 10518
Outlier documents: 2167
Outlier percentage: 20.6 %


In [11]:
#TOPIC FREQUENCY
topic_frequency = improved_topic_model.get_topic_freq()

print(topic_frequency)

    Topic  Count
0      -1   2167
4       2   1577
2       7   1225
7       4    641
1      11    595
5       6    504
19     20    384
15     10    315
3      16    313
11     19    244
6       5    233
10      8    209
18     25    202
20     21    177
17      0    172
12     15    164
24     23    159
9      14    157
21     17    144
16     13    128
26      9    124
14     22    120
8      24    118
22     18    117
25      1    115
23     12    108
13      3    106


In [12]:
#REMOVE OUTLIER FOR THE TOPIC FREQUENCY TABLE
topic_frequency_no_outliers = topic_frequency[
    topic_frequency["Topic"] != -1
].copy()

topic_frequency_no_outliers.head(10)

,Topic,Count
4,2,1577
2,7,1225
7,4,641
1,11,595
5,6,504
19,20,384
15,10,315
3,16,313
11,19,244
6,5,233


In [13]:
#Display the top 10 largest topics
top_topics = topic_frequency_no_outliers.head(10)

print(top_topics)

    Topic  Count
4       2   1577
2       7   1225
7       4    641
1      11    595
5       6    504
19     20    384
15     10    315
3      16    313
11     19    244
6       5    233


In [14]:
#Get the words for each topic
for topic_id in range(num_topics):
    
    words = improved_topic_model.get_topic(topic_id)
    
    print(f"\nTopic {topic_id}:")
    print([word for word, score in words[:10]])


Topic 0:
['watch', 'band', 'wrist', 'wear', 'track', 'smart', 'app', 'step', 'feature', 'phone']

Topic 1:
['alexa', 'echo', 'dot', 'ad', 'speaker', 'clock', 'music', 'play', 'smart', 'sound']

Topic 2:
['sound', 'headphone', 'speaker', 'ear', 'music', 'good', 'phone', 'quality', 'great', 'sound quality']

Topic 3:
['fan', 'cool', 'air', 'pad', 'laptop', 'quiet', 'blow', 'dust', 'little', 'temperature']

Topic 4:
['tv', 'remote', 'antenna', 'channel', 'work', 'picture', 'player', 'great', 'button', 'good']

Topic 5:
['router', 'network', 'wifi', 'device', 'wireless', 'signal', 'internet', 'extender', 'speed', 'connect']

Topic 6:
['star', 'star great', 'star work', 'star good', 'star love', 'great', 'work', 'star nice', 'love', 'great star']

Topic 7:
['cable', 'charge', 'cord', 'charger', 'battery', 'work', 'plug', 'great', 'long', 'outlet']

Topic 8:
['laptop', 'monitor', 'screen', 'graphic', 'window', 'gaming', 'game', 'use', 'work', 'ram']

Topic 9:
['port', 'hub', 'adapter', 'usb

In [15]:
#Create a topic-word table
topic_words_table = []

for topic_id in range(num_topics):
    
    words = improved_topic_model.get_topic(topic_id)
    
    top_words = [
        word for word, score in words[:10]
    ]
    
    topic_words_table.append({
        "Topic": topic_id,
        "Top Words": ", ".join(top_words)
    })

topic_words_df = pd.DataFrame(topic_words_table)

topic_words_df.head(10)



,Topic,Top Words
0,0,"watch, band, wrist, wear, track, smart, app, s..."
1,1,"alexa, echo, dot, ad, speaker, clock, music, p..."
2,2,"sound, headphone, speaker, ear, music, good, p..."
3,3,"fan, cool, air, pad, laptop, quiet, blow, dust..."
4,4,"tv, remote, antenna, channel, work, picture, p..."
5,5,"router, network, wifi, device, wireless, signa..."
6,6,"star, star great, star work, star good, star l..."
7,7,"cable, charge, cord, charger, battery, work, p..."
8,8,"laptop, monitor, screen, graphic, window, gami..."
9,9,"port, hub, adapter, usb, plug, dock, power, wo..."


In [16]:
#CALCULATE BERTOPIC c_v coherence 
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
texts_for_coherence = [
    document.lower().split()
    for document in documents
]

dictionary = Dictionary(
    texts_for_coherence
)

In [17]:
#Extract BERTopic words
topic_words = []

for topic_id in range(num_topics):
    
    words = improved_topic_model.get_topic(topic_id)
    
    words = [
        word for word, score in words[:10]
    ]
    
    topic_words.append(words)

print("Number of topic word lists:", len(topic_words))

Number of topic word lists: 26


In [18]:
#Calculate c_v coherence
coherence_model = CoherenceModel(
    topics=topic_words,
    texts=texts_for_coherence,
    dictionary=dictionary,
    coherence="c_v"
)

bert_coherence = coherence_model.get_coherence()

print(
    "BERTopic c_v Coherence for topics:",
    round(bert_coherence, 4)
)

BERTopic c_v Coherence for topics: 0.5702


In [19]:
#BERTopic EVALUVATION SUMMARY
bert_summary = pd.DataFrame({
    "Metric": [
        "Number of Topics",
        "Coherence",
        "Outlier Documents",
        "Outlier Percentage",
        "Training Time (seconds)",
        "Training Time (minutes)"
    ],
    
    "Value": [
        num_topics,
        round(bert_coherence, 4),
        outlier_count,
        round(outlier_percentage, 2),
        round(training_time, 2),
        round(training_time / 60, 2)
    ]
})

print(bert_summary)
bert_summary.to_csv(
    "../results/BERTopic_Evaluation_Summary.csv",
    index=False
)

                    Metric      Value
0         Number of Topics    26.0000
1                Coherence     0.5702
2        Outlier Documents  2167.0000
3       Outlier Percentage    20.6000
4  Training Time (seconds)   312.1800
5  Training Time (minutes)     5.2000


In [20]:
fig = improved_topic_model.visualize_barchart(
    top_n_topics=10,
    n_words=8
)

fig.show()
fig.write_html(
    "BERTopic_Top10_Barchart.html"
)

In [21]:
fig = improved_topic_model.visualize_topics()

fig.show()
fig.write_html(
    "BERTopic_Intertopic_Distance.html"
)

In [22]:
fig = improved_topic_model.visualize_heatmap(
    top_n_topics=20
)

fig.show()
fig.write_html(
    "BERTopic_Heatmap.html"
)

In [23]:
hierarchical_topics = improved_topic_model.hierarchical_topics(
    documents
)

print(hierarchical_topics.head())
fig = improved_topic_model.visualize_hierarchy(
    hierarchical_topics=hierarchical_topics
)

fig.show()
fig.write_html(
    "BERTopic_Hierarchy.html"
)

100%|██████████| 25/25 [00:00<00:00, 75.50it/s]


   Parent_ID                      Parent_Name  \
24        50        work_great_good_use_sound   
23        49       star_case_great_ipad_cover   
22        48        work_great_good_use_sound   
21        47  bag_mouse_laptop_keyboard_stand   
20        46        work_sound_good_great_use   

                                               Topics Child_Left_ID  \
24  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...            48   
23                        [6, 14, 17, 19, 20, 24, 25]            38   
22  [0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 15...            46   
21                        [3, 13, 15, 16, 18, 22, 23]            43   
20           [0, 1, 2, 4, 5, 7, 8, 9, 10, 11, 12, 21]            41   

                    Child_Left_Name Child_Right_ID  \
24        work_great_good_use_sound             49   
23         case_ipad_cover_fit_love             45   
22        work_sound_good_great_use             47   
21  bag_mouse_laptop_keyboard_stand             42   
20  soun

In [24]:
# ============================================================
# BERTopic SCALABILITY TEST
# ============================================================

import time
import pandas as pd
import numpy as np

from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel


# ------------------------------------------------------------
# 1. Dataset sizes to test
# ------------------------------------------------------------

dataset_sizes = [2000, 5000, 7500, 10518]

scalability_results = []


# ------------------------------------------------------------
# 2. Run BERTopic for each dataset size
# ------------------------------------------------------------

for sample_size in dataset_sizes:

    print("\n" + "=" * 70)
    print(f"BERTopic SCALABILITY TEST: {sample_size} REVIEWS")
    print("=" * 70)

    # Take first N reviews
    sample_documents = documents[:sample_size]

    print(f"Number of documents: {len(sample_documents)}")


    # --------------------------------------------------------
    # Create the same vectorizer as the improved model
    # --------------------------------------------------------

    scalability_vectorizer = CountVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=5
    )


    # --------------------------------------------------------
    # Create BERTopic model using the SAME configuration
    # --------------------------------------------------------

    scalability_model = BERTopic(

        embedding_model=embedding_model,

        vectorizer_model=scalability_vectorizer,

        language="english",

        calculate_probabilities=True,

        nr_topics=30,

        

        umap_model=umap_model,

        hdbscan_model= hdbscan_model,

        verbose=False
    )


    # --------------------------------------------------------
    # Start timer
    # --------------------------------------------------------

    start_time = time.time()


    # Train model
    sample_topics, sample_probabilities = (
        scalability_model.fit_transform(sample_documents)
    )


    # End timer
    training_time = time.time() - start_time


    # --------------------------------------------------------
    # Topic information
    # --------------------------------------------------------

    sample_topic_info = (
        scalability_model.get_topic_info()
    )


    # Exclude outlier topic (-1)
    valid_topics = sample_topic_info[
        sample_topic_info["Topic"] != -1
    ]


    number_of_topics = len(valid_topics)


    # --------------------------------------------------------
    # Outlier calculation
    # --------------------------------------------------------

    outlier_count = sum(
        1 for topic in sample_topics
        if topic == -1
    )

    outlier_percentage = (
        outlier_count / len(sample_documents)
    ) * 100


    # --------------------------------------------------------
    # Vocabulary size
    # --------------------------------------------------------

    vocabulary_size = len(
        scalability_model.vectorizer_model.vocabulary_
    )


    # --------------------------------------------------------
    # Prepare documents for coherence
    # --------------------------------------------------------

    tokenized_documents = [
        document.lower().split()
        for document in sample_documents
    ]


    # Create Gensim dictionary
    coherence_dictionary = Dictionary(
        tokenized_documents
    )


    # --------------------------------------------------------
    # Extract top words for each topic
    # --------------------------------------------------------

    topic_words = []

    for topic_id in valid_topics["Topic"]:

        words = scalability_model.get_topic(
            topic_id
        )

        if words is not None:

            top_words = [
                word
                for word, score in words[:10]
            ]

            topic_words.append(top_words)


    # --------------------------------------------------------
    # Calculate c_v coherence
    # --------------------------------------------------------

    coherence_score = np.nan

    if len(topic_words) > 0:

        coherence_model = CoherenceModel(

            topics=topic_words,

            texts=tokenized_documents,

            dictionary=coherence_dictionary,

            coherence="c_v"
        )

        coherence_score = (
            coherence_model.get_coherence()
        )


    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    scalability_results.append({

        "Dataset Size": len(sample_documents),

        "Vocabulary Size": vocabulary_size,

        "Number of Topics": number_of_topics,

        "Training Time (seconds)": training_time,

        "Training Time (minutes)": training_time / 60,

        "Coherence Score (c_v)": coherence_score,

        "Outlier Documents": outlier_count,

        "Outlier Percentage": outlier_percentage
    })


    # --------------------------------------------------------
    # Print results
    # --------------------------------------------------------

    print(f"Vocabulary size:       {vocabulary_size}")
    print(f"Number of topics:      {number_of_topics}")
    print(f"Training time:         {training_time:.2f} seconds")
    print(f"Training time:         {training_time / 60:.2f} minutes")
    print(f"Coherence (c_v):       {coherence_score:.4f}")
    print(f"Outlier documents:     {outlier_count}")
    print(f"Outlier percentage:    {outlier_percentage:.2f}%")


# ============================================================
# 3. Create scalability results table
# ============================================================

scalability_df = pd.DataFrame(
    scalability_results
)


print("\n")
print("=" * 90)
print("BERTopic SCALABILITY RESULTS")
print("=" * 90)

print(
    scalability_df.to_string(
        index=False
    )
)


BERTopic SCALABILITY TEST: 2000 REVIEWS
Number of documents: 2000
Vocabulary size:       903
Number of topics:      7
Training time:         76.34 seconds
Training time:         1.27 minutes
Coherence (c_v):       0.4357
Outlier documents:     347
Outlier percentage:    17.35%

BERTopic SCALABILITY TEST: 5000 REVIEWS
Number of documents: 5000
Vocabulary size:       3110
Number of topics:      13
Training time:         178.55 seconds
Training time:         2.98 minutes
Coherence (c_v):       0.5350
Outlier documents:     653
Outlier percentage:    13.06%

BERTopic SCALABILITY TEST: 7500 REVIEWS
Number of documents: 7500
Vocabulary size:       4861
Number of topics:      15
Training time:         208.79 seconds
Training time:         3.48 minutes
Coherence (c_v):       0.5421
Outlier documents:     1264
Outlier percentage:    16.85%

BERTopic SCALABILITY TEST: 10518 REVIEWS
Number of documents: 10518
Vocabulary size:       6150
Number of topics:      26
Training time:         225.94 sec